In [ ]:
from notebook.services.config import ConfigManagercm = ConfigManager()cm.update('livereveal', {'width': 1920, 'height': 1080, 'scroll': True})

# Week 13: Monday, AST 5011: Astrophysical Systems

## Galactic Dynamics & Scaling Relations

### Michael Coughlin

**Reference:** Cimatti, Fraternali & Nipoti, Ch. 11

With material from Benedikt Diemer (UMD).

In [ ]:
import numpy as npimport scipyimport matplotlib.pyplot as pltfrom colossus.cosmology import cosmology%matplotlib inline%config InlineBackend.figure_format = 'retina'cosmo = cosmology.setCosmology('planck18')

import os
from astropy.io import fits

# Data directory (same folder as this notebook)
_data_dir = os.path.join(os.getcwd(), 'data')

# Solar absolute magnitudes (AB, Blanton & Roweis 2007)
solar_mag = {'u': 6.55, 'g': 5.12, 'r': 4.68, 'i': 4.57, 'z': 4.54}

def loadUPennSubset():
    """Load pre-extracted UPenn catalog subset."""
    data = np.load(os.path.join(_data_dir, 'upenn_subset.npz'), allow_pickle=True)
    return data['data']

def loadBradford():
    """Load Bradford+2016 Tully-Fisher data."""
    fn = os.path.join(_data_dir, 'bradford_2016_figure1.fits')
    with fits.open(fn) as hdul:
        return hdul[1].data

def selectEllipticals(d, min_prob=0.7, mag_cut=16.0):
    """Select ellipticals (Ell + S0) from UPenn catalog."""
    p_ell = np.maximum(d['probaEll_h11'], d['probaS0_h11'])
    p_dis = np.maximum(d['probaSab_h11'], d['probaScd_h11'])
    mask = (p_ell > min_prob) & (p_ell > p_dis)
    if mag_cut is not None:
        mask &= (d['m_tot_r'] < mag_cut)
    return d[mask]

def absoluteMagnitude(m_app, extinction, DM, kcorr):
    """Absolute magnitude from apparent magnitude, extinction, distance modulus, k-correction."""
    return m_app - extinction - DM - kcorr

def logLuminosity(M_abs, band='r'):
    """Log10 luminosity in solar luminosities from absolute magnitude."""
    return 0.4 * (solar_mag[band] - M_abs)

def lineFit(x, a, b):
    """Linear model y = a + b*x."""
    return a + b * x

def fitPlaneSVD(x, y, z):
    """Fit a plane to 3D data using SVD. Returns a, b, c such that x = a*y + b*z + c."""
    XYZ = np.stack((x, y, z)).T
    rows, cols = XYZ.shape
    p = np.ones((rows, 1))
    AB = np.hstack([XYZ, p])
    u, d, v = np.linalg.svd(AB, 0)
    B = v[3, :]
    nn = np.linalg.norm(B[0:3])
    B = B / nn
    return -B[1]/B[0], -B[2]/B[0], -B[3]/B[0]

## Scaling Relations

Galaxies obey remarkably tight scaling relations between their structural properties (size, luminosity, velocity). These relations encode information about the physics of galaxy formation.

![Galaxy Formation](figures/galaxy_formation.png)

Disk galaxies:
- Tully-Fisher relation: $L \propto V_{\text{rot}}^{\alpha}$ with $\alpha \approx 3$-$4$
- Relates luminosity (or stellar/baryonic mass) to rotation velocity

Elliptical galaxies:
- Faber-Jackson relation: $L \propto \sigma^{\gamma}$ with $\gamma \approx 4$
- Relates luminosity to velocity dispersion

Fundamental Plane:
- Ellipticals lie on a 2D surface in the 3D space of ($R_e$, $\mu_e$, $\sigma$)
- Tighter than Faber-Jackson alone (reduces scatter from ~0.4 dex to ~0.1 dex)

## Disk Galaxies: The Tully-Fisher Relation

The Tully-Fisher relation (TFR) connects the rotational velocity of a disk galaxy (measured from HI 21-cm line widths) to its luminosity or stellar mass.

Physical origin: for a self-gravitating exponential disk in a dark matter halo,

$$V_{\text{rot}}^2 \sim \frac{G M_{\text{total}}}{R}$$

If $M_*/L$ is roughly constant, then $L \propto V_{\text{rot}}^4$ (the "naive" prediction).

The baryonic Tully-Fisher relation (BTFR) uses total baryonic mass (stars + gas) rather than just luminosity, and is even tighter: $M_{\text{bar}} \propto V_{\text{rot}}^{\alpha}$ with $\alpha \approx 3.5$-$4$.

## Exercise 1: The Tully-Fisher RelationUsing rotation velocities and stellar masses from Bradford et al. (2016):1. Plot the stellar-mass TFR: $M_*$ vs $V_{\text{rot}}$2. Fit a power law: $\log_{10} M_* = a + b \cdot \log_{10} V_{\text{rot}}$3. Plot the baryonic TFR: $M_{\text{bar}}$ vs $V_{\text{rot}}$ and fit

In [ ]:
# Exercise 1: Tully-Fisher Relation from Bradford+2016

brad =  # FILL IN: load Bradford data

# Extract columns
M_bar = brad['MBARYON'] + 0.1238  # correct for Helium
M_bar_err = brad['MBARYON_ERR']
V_rot = brad['VW20I']
V_rot_err = brad['VW20I_ERR']
M_star = brad['MSTAR']
M_star_err = brad['MSTAR_ERR']

sig_min = 20.0
sig_max = 500.0
plt_x = np.linspace(np.log10(sig_min), np.log10(sig_max), 5)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Left: Stellar mass TFR
ax1 = axes[0]
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlim(sig_min, sig_max)
ax1.set_ylim(1e6, 1e12)
ax1.set_xlabel(r'$V_{\rm rot}\ (\rm km/s)$')
ax1.set_ylabel(r'$M_*\ (M_\odot)$')

ax1.errorbar(V_rot, 10**M_star, xerr=V_rot_err,
             yerr=[10**(M_star - M_star_err), 10**(M_star + M_star_err)],
             fmt='o', ms=1.5, color='C0', ecolor='gray', lw=0.1)

# FILL IN: fit power law to stellar TFR
params_star, _ =  # FILL IN
plt_y = lineFit(plt_x, params_star[0], params_star[1])
ax1.plot(10**plt_x, 10**plt_y, 'C1-', lw=2)
ax1.set_title(f'Stellar TFR: slope = {params_star[1]:.2f}')

# Right: Baryonic TFR
ax2 = axes[1]
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim(sig_min, sig_max)
ax2.set_ylim(1e7, 1e12)
ax2.set_xlabel(r'$V_{\rm rot}\ (\rm km/s)$')
ax2.set_ylabel(r'$M_{\rm bar}\ (M_\odot)$')

ax2.errorbar(V_rot, 10**M_bar, xerr=V_rot_err,
             yerr=[10**(M_bar - M_bar_err), 10**(M_bar + M_bar_err)],
             fmt='o', ms=1.5, color='C0', ecolor='gray', lw=0.1)

# FILL IN: fit power law to baryonic TFR
params_bar, _ =  # FILL IN
plt_y = lineFit(plt_x, params_bar[0], params_bar[1])
ax2.plot(10**plt_x, 10**plt_y, 'C1-', lw=2)
ax2.set_title(f'Baryonic TFR: slope = {params_bar[1]:.2f}')

plt.tight_layout()
plt.show()

print(f'Stellar TFR: log M* = {params_star[0]:.2f} + {params_star[1]:.2f} * log V_rot')
print(f'Baryonic TFR: log M_bar = {params_bar[0]:.2f} + {params_bar[1]:.2f} * log V_rot')

## Demonstration: TFR Scatter -- Stellar vs. Baryonic

A key test of the Tully-Fisher relation is its scatter: the baryonic TFR should be tighter than the stellar TFR because $M_{\rm bar}$ is a more fundamental quantity (total baryons) than $M_*$ alone. We compare the residual distributions and how the scatter varies with rotation velocity.

In [ ]:
# TFR residuals: stellar vs baryonic
log_Vrot = np.log10(V_rot)
resid_star = M_star - lineFit(log_Vrot, params_star[0], params_star[1])
resid_bar = M_bar - lineFit(log_Vrot, params_bar[0], params_bar[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: residual distributions
ax1.hist(resid_star, bins=30, alpha=0.6, color='C0', density=True,
         label=r'Stellar ($\sigma$ = %.2f dex)' % np.std(resid_star))
ax1.hist(resid_bar, bins=30, alpha=0.6, color='C1', density=True,
         label=r'Baryonic ($\sigma$ = %.2f dex)' % np.std(resid_bar))
ax1.axvline(0, ls='--', color='gray', lw=0.8)
ax1.set_xlabel(r'$\Delta \log_{10} M$ (residual from fit)')
ax1.set_ylabel('PDF')
ax1.legend(fontsize=9)
ax1.set_title('TFR Scatter: Stellar vs. Baryonic')

# Right: scatter in bins of V_rot
n_bins = 8
bin_edges = np.linspace(log_Vrot.min() + 0.05, log_Vrot.max() - 0.05, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
scat_star = np.zeros(n_bins)
scat_bar = np.zeros(n_bins)
for k in range(n_bins):
    mask_bin = (log_Vrot >= bin_edges[k]) & (log_Vrot < bin_edges[k + 1])
    if np.sum(mask_bin) > 5:
        scat_star[k] = np.std(resid_star[mask_bin])
        scat_bar[k] = np.std(resid_bar[mask_bin])
mask_v = (scat_star > 0)
ax2.plot(10**bin_centers[mask_v], scat_star[mask_v], 'o-', color='C0', lw=2, label='Stellar')
ax2.plot(10**bin_centers[mask_v], scat_bar[mask_v], 's--', color='C1', lw=2, label='Baryonic')
ax2.set_xscale('log')
ax2.set_xlabel(r'$V_{\rm rot}\ (\rm km/s)$')
ax2.set_ylabel('Scatter (dex)')
ax2.legend(fontsize=9)
ax2.set_title('TFR Scatter vs. Rotation Velocity')

plt.tight_layout()
plt.show()

print(f'Overall stellar TFR scatter: {np.std(resid_star):.3f} dex')
print(f'Overall baryonic TFR scatter: {np.std(resid_bar):.3f} dex')

## Elliptical Galaxies

Elliptical galaxies are pressure-supported systems (random motions, not rotation). Their kinematic tracer is the velocity dispersion $\sigma$, measured from spectral line broadening.

![Disky vs Boxy Ellipticals](figures/disky_boxy.png)

We use the UPenn photometric catalog (Meert et al. 2015) combined with SDSS spectroscopy to study elliptical galaxy scaling relations. Morphological classifications come from Huertas-Company et al. (2011).

Selection: we require $p_{\text{Ell}} + p_{\text{S0}} > 0.7$ and $> p_{\text{Sab}} + p_{\text{Scd}}$ (confident elliptical/S0 classification).

## Demonstration: Scaling Relations as Distance Indicators

The Tully-Fisher and Fundamental Plane relations are among the most important distance indicators in extragalactic astronomy. The principle is simple:

1. Measure a distance-independent quantity (rotation velocity $V_{\rm rot}$ from HI line width, or $\sigma$ and $\mu_e$ from spectroscopy/imaging)
2. Use the scaling relation to predict the intrinsic luminosity or physical size
3. Compare to the observed flux or angular size to get the luminosity distance

The TFR was used in the HST Key Project to measure $H_0$, and the FP is used to measure peculiar velocities that map the cosmic flow field. The precision of these methods depends directly on the scatter in the scaling relation.

In [ ]:
# Distance indicator demonstration: TFR and FP precision

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Left: TFR distance estimate for a mock galaxy
ax1 = axes[0]
V_rot_mock = np.array([80, 120, 180, 250])  # km/s
log_M_pred = lineFit(np.log10(V_rot_mock), params_bar[0], params_bar[1])
scatter_tfr = np.std(resid_bar)

ax1.errorbar(V_rot_mock, log_M_pred, yerr=scatter_tfr,
             fmt='s', ms=8, color='C0', capsize=5, lw=2)
V_plot = np.linspace(50, 350, 100)
ax1.plot(V_plot, lineFit(np.log10(V_plot), params_bar[0], params_bar[1]),
         '--', color='C1', lw=1.5, alpha=0.7)

# Show distance uncertainty from mass uncertainty
for i, v in enumerate(V_rot_mock):
    # delta_DM = 2.5 * delta_log_L => delta_d/d = 10^(0.2*delta_DM) - 1
    frac_dist_err = (10**(0.2 * 2.5 * scatter_tfr) - 1) * 100
    ax1.annotate(f'{frac_dist_err:.0f}% dist err',
                 xy=(v, log_M_pred[i] + scatter_tfr + 0.1),
                 fontsize=8, ha='center', color='gray')

ax1.set_xscale('log')
ax1.set_xlabel(r'$V_{\rm rot}\ (\rm km/s)$')
ax1.set_ylabel(r'$\log_{10}\ M_{\rm bar}\ (M_\odot)$')
ax1.set_title('TFR as Distance Indicator')
ax1.set_xlim(50, 350)

# Right: comparison of distance indicator precision
ax2 = axes[1]
methods = ['TFR\n(stellar)', 'TFR\n(baryonic)', 'Faber-\nJackson', 'Fundamental\nPlane', 'SN Ia']
scatters_mag = [np.std(resid_star) * 2.5,
                np.std(resid_bar) * 2.5,
                0.4 / params_fj[1] * 2.5,
                scatter * 2.5 / 0.6,  # convert R_e scatter to mag via size-L relation
                0.15]  # typical SN Ia scatter
dist_errs = [(10**(0.2 * s) - 1) * 100 for s in scatters_mag]

colors_bar = ['C0', 'C1', 'C2', 'C3', 'C4']
bars = ax2.bar(methods, dist_errs, color=colors_bar, alpha=0.7, edgecolor='black', lw=0.5)
ax2.set_ylabel('Distance uncertainty (%)')
ax2.set_title('Precision of Distance Indicators')
for bar, err in zip(bars, dist_errs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{err:.0f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Exercise 2: The Faber-Jackson RelationFor elliptical galaxies from the UPenn catalog:1. Compute absolute r-band luminosity $L_r$ from apparent magnitude, extinction, distance modulus, and k-correction2. Plot $\log_{10} \sigma$ vs $\log_{10} L_r$ as a 2D histogram3. Fit the Faber-Jackson relation $\log_{10} \sigma = a + b \cdot \log_{10} L_r$4. Compare the slope to the canonical $L \propto \sigma^4$ prediction

In [ ]:
# Exercise 2: Faber-Jackson Relation

# Load and select ellipticals
d = loadUPennSubset()

ell =  # FILL IN: select elliptical galaxies

M_r =    # FILL IN: compute absolute magnitude
log_L =  # FILL IN: compute log luminosity
sigma_v = ell['veldisp']

# Clean: remove NaN and zero-sigma entries
mask = np.isfinite(log_L) & (sigma_v > 0.0)
x = log_L[mask]
y = np.log10(sigma_v[mask])
weights = ell['1/Vmax'][mask] if '1/Vmax' in ell.dtype.names else np.ones(np.sum(mask))

# Fit
params_fj, _ = scipy.optimize.curve_fit(lineFit, x, y, p0=[1.0, 0.25],
                                         sigma=1.0/weights)
print(f'Faber-Jackson fit: log sigma = {params_fj[0]:.2f} + {params_fj[1]:.3f} * log L')
print(f'Equivalent: L ~ sigma^{1/params_fj[1]:.1f}')

# Plot
x_lo, x_hi = 9.7, 11.6
y_lo, y_hi = 1.8, 2.6

cmap = plt.get_cmap('Blues')
cmap.set_under('#FFFFFF')

plt.figure(figsize=(5, 4))
plt.xlabel(r'$\log_{10}\ L_r\ (L_\odot)$')
plt.ylabel(r'$\log_{10}\ \sigma\ (\rm km/s)$')
plt.xlim(x_lo, x_hi)
plt.ylim(y_lo, y_hi)

hist, _, _ = np.histogram2d(x, y, bins=(40, 40),
                             range=[[x_lo, x_hi], [y_lo, y_hi]],
                             weights=weights, density=True)
hist = np.log10(hist.T[::-1] + 1.0)
plt.imshow(hist, extent=[x_lo, x_hi, y_lo, y_hi], interpolation='nearest',
           aspect='auto', cmap=cmap, vmin=1e-3, vmax=np.max(hist))

# Fit line
plt_x = np.linspace(x_lo, x_hi, 5)
plt_y = lineFit(plt_x, params_fj[0], params_fj[1])
plt.plot(plt_x, plt_y, '--', lw=1.5, color='C1', label=f'Fit: $L \\propto \\sigma^{{{1/params_fj[1]:.1f}}}$')

# Median and scatter
med_args = dict(range=(x_lo, x_hi), bins=20)
med, bin_edges, _ = scipy.stats.binned_statistic(x, y, statistic='median', **med_args)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) * 0.5
plt.plot(bin_centers, med, '-', lw=1.0, color='C1')

plt.legend(fontsize=10)
plt.colorbar(label=r'$\log_{10} ({\rm density} + 1)$')
plt.title('Faber-Jackson Relation')
plt.tight_layout()
plt.show()

## Demonstration: Size-Luminosity Relation

In addition to the Faber-Jackson relation, elliptical galaxies show a strong correlation between their effective radius $R_e$ and luminosity $L$. More luminous ellipticals are larger, roughly as $R_e \propto L^{0.6}$. This relation is a projection of the fundamental plane and connects to the physics of how ellipticals grow (dry mergers increase size more than luminosity).

In [ ]:
# Size-luminosity relation for ellipticals
R_eff_arcsec_sl = ell['r_tot_r']
R_eff_kpc_sl = R_eff_arcsec_sl * ell['kpc_per_arcsec_r']
M_r_sl = absoluteMagnitude(ell['m_tot_r'], ell['extinction_r'],
                               ell['DM'], ell['kcorr_r'])
log_L_sl = logLuminosity(M_r_sl, band='r')

mask_sl = np.isfinite(log_L_sl) & (R_eff_kpc_sl > 0)
x_sl = log_L_sl[mask_sl]
y_sl = np.log10(R_eff_kpc_sl[mask_sl])

cmap_sl = plt.get_cmap('Blues')
cmap_sl.set_under('#FFFFFF')

fig, ax = plt.subplots(figsize=(5, 4.5))
xl, xh = 9.5, 11.8
yl, yh = -0.5, 2.0

hist_sl, _, _ = np.histogram2d(x_sl, y_sl, bins=(40, 40),
                                range=[[xl, xh], [yl, yh]], density=True)
hist_sl = np.log10(hist_sl.T[::-1] + 1.0)
ax.imshow(hist_sl, extent=[xl, xh, yl, yh], interpolation='nearest',
          aspect='auto', cmap=cmap_sl, vmin=1e-3, vmax=np.max(hist_sl))

# Median relation
med_sl, edges_sl, _ = scipy.stats.binned_statistic(
    x_sl, y_sl, statistic='median', range=(xl + 0.2, xh - 0.2), bins=15)
cen_sl = 0.5 * (edges_sl[:-1] + edges_sl[1:])
ax.plot(cen_sl, med_sl, '-', lw=1.5, color='C1', label='Median')

# Fit
params_sl, _ = scipy.optimize.curve_fit(lineFit, x_sl, y_sl, p0=[-5.0, 0.6])
fit_x = np.linspace(xl, xh, 5)
ax.plot(fit_x, lineFit(fit_x, *params_sl), '--', lw=1.5, color='C1',
        label=f'Fit: slope = {params_sl[1]:.2f}')

ax.set_xlabel(r'$\log_{10}\ L_r\ (L_\odot)$')
ax.set_ylabel(r'$\log_{10}\ R_e\ (\rm kpc)$')
ax.set_xlim(xl, xh)
ax.set_ylim(yl, yh)
ax.legend(fontsize=9)
ax.set_title('Size-Luminosity Relation (Ellipticals)')
plt.colorbar(ax.images[0], ax=ax, label=r'$\log_{10}({\rm density} + 1)$')
plt.tight_layout()
plt.show()

print(f'Size-luminosity fit: log R_e = {params_sl[0]:.2f} + {params_sl[1]:.2f} * log L_r')

## The Fundamental Plane

The Faber-Jackson relation has significant scatter (~0.4 dex in $\sigma$). Much of this scatter is reduced by considering a third parameter: the effective radius $R_e$.

Elliptical galaxies lie on a fundamental plane in the space of $(\log R_e, \mu_e, \log \sigma)$:

$$\log_{10} R_e = a \cdot \mu_e + b \cdot \log_{10} \sigma + c$$

where $\mu_e$ is the mean surface brightness within $R_e$. Classic results:
- Djorgovsky & Davis (1987): $a \approx 0.36$, $b \approx 1.39$
- Dressler et al. (1987): $a \approx 0.33$, $b \approx 1.33$

The "tilt" of the fundamental plane away from the virial prediction ($a = 0.4$, $b = 2$) encodes information about how $M/L$ varies along the plane.

## Exercise 3: The Fundamental PlaneUsing the elliptical galaxy sample:1. Compute $R_e$ (kpc), surface brightness $\mu_e$, and $\sigma$2. Fit the fundamental plane using SVD3. Compare coefficients to Djorgovsky & Davis (1987) and Dressler et al. (1987)

In [ ]:
# Exercise 3: Fundamental Plane

# Use the elliptical sample from Exercise 2 with stricter cuts
M_r_all = absoluteMagnitude(ell['m_tot_r'], ell['extinction_r'], ell['DM'], ell['kcorr_r'])
log_L_all = logLuminosity(M_r_all, band='r')
sigma_all = ell['veldisp']
R_eff_arcsec = ell['r_tot_r']

# FILL IN: apply quality cuts (R_eff >= 1 arcsec, sigma >= 200 km/s, valid L)
mask_fp =  # FILL IN

R_eff_arcsec_m = R_eff_arcsec[mask_fp]
R_eff_kpc = R_eff_arcsec_m * ell['kpc_per_arcsec_r'][mask_fp]

# Surface brightness
mu_e = ell['m_tot_r'][mask_fp] + 2.5 * np.log10(2.0 * np.pi * R_eff_arcsec_m**2)

log_R = np.log10(R_eff_kpc)
log_sigma = np.log10(sigma_all[mask_fp])

# FILL IN: fit fundamental plane
a_fp, b_fp, c_fp =  # FILL IN

print(f'Fundamental Plane: log R_e = {a_fp:.2f} * mu_e + {b_fp:.2f} * log sigma + {c_fp:.2f}')
print(f'Djorgovsky & Davis 1987: a = 0.36, b = 1.39')
print(f'Dressler et al. 1987:    a = 0.33, b = 1.33')

# Plot edge-on view of the FP
# Predicted log R from the plane
log_R_pred = a_fp * mu_e + b_fp * log_sigma + c_fp

plt.figure(figsize=(5, 4.5))
plt.scatter(log_R, log_R_pred, s=0.3, alpha=0.3, c='C0')
plt.plot([log_R.min(), log_R.max()], [log_R.min(), log_R.max()], 'r--', lw=1)
plt.xlabel(r'$\log_{10}\ R_e\ (\rm kpc)$ (observed)')
plt.ylabel(r'$\log_{10}\ R_e\ (\rm kpc)$ (predicted)')
plt.title('Fundamental Plane: Observed vs Predicted')

scatter = np.std(log_R - log_R_pred)
plt.text(0.05, 0.92, f'scatter = {scatter:.3f} dex', transform=plt.gca().transAxes, fontsize=11)

plt.tight_layout()
plt.show()

print(f'\nScatter in log R_e: {scatter:.3f} dex')
print(f'N galaxies in FP fit: {np.sum(mask_fp)}')

## Demonstration: The Fundamental Plane Tilt

The virial theorem predicts $R_e \propto \sigma^2 / I_e$, implying FP coefficients $a = 0.4$, $b = 2$. The observed tilt ($a \approx 0.3$, $b \approx 1.4$) means that $M/L$ must vary systematically along the plane. We can verify this by computing the dynamical mass $M_{\rm dyn} = k\,\sigma^2\,R_e / G$ (with $k \approx 5$) and comparing to the luminosity.

In [ ]:
# FP tilt: dynamical M/L vs luminosity
# Dynamical mass: M_dyn = k * sigma^2 * R_e / G (k ~ 5 for de Vaucouleurs profiles)
G_pc = 4.301e-3  # G in (km/s)^2 pc Msun^-1

sigma_fp = sigma_all[mask_fp]
R_eff_pc_fp = R_eff_kpc * 1e3  # kpc -> pc
L_fp = 10**log_L_all[mask_fp]

M_dyn = 5.0 * sigma_fp**2 * R_eff_pc_fp / G_pc  # Msun
log_ML = np.log10(M_dyn / L_fp)
log_L_fp = np.log10(L_fp)

mask_ml = np.isfinite(log_ML) & np.isfinite(log_L_fp) & (log_ML > -1) & (log_ML < 3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: M_dyn/L vs L
ax1.plot(log_L_fp[mask_ml], log_ML[mask_ml], '.', ms=0.5, alpha=0.2, color='C0')
med_ml, edges_ml, _ = scipy.stats.binned_statistic(
    log_L_fp[mask_ml], log_ML[mask_ml], statistic='median',
    range=(9.8, 11.5), bins=15)
cen_ml = 0.5 * (edges_ml[:-1] + edges_ml[1:])
ax1.plot(cen_ml, med_ml, 'o-', color='C1', lw=2, ms=5, label='Median')
ax1.set_xlabel(r'$\log_{10}\ L_r\ (L_\odot)$')
ax1.set_ylabel(r'$\log_{10}\ (M_{\rm dyn} / L_r)$')
ax1.set_xlim(9.7, 11.6)
ax1.legend(fontsize=10)
ax1.set_title(r'FP Tilt: $M/L$ Increases with Luminosity')

# Right: M_dyn vs L with 1:1 line
ax2.plot(log_L_fp[mask_ml], np.log10(M_dyn[mask_ml]), '.', ms=0.5, alpha=0.2, color='C0')
ax2.plot([9, 13], [9, 13], 'k--', lw=0.8, label=r'$M_{\rm dyn}/L = 1$')
ax2.plot([9, 13], [9.5, 13.5], ':', color='gray', lw=0.8, label=r'$M_{\rm dyn}/L = 3$')
ax2.set_xlabel(r'$\log_{10}\ L_r\ (L_\odot)$')
ax2.set_ylabel(r'$\log_{10}\ M_{\rm dyn}\ (M_\odot)$')
ax2.set_xlim(9.7, 11.6)
ax2.set_ylim(9.7, 12.5)
ax2.legend(fontsize=9)
ax2.set_title('Dynamical Mass vs. Luminosity')

plt.tight_layout()
plt.show()

## Summary

1. The Tully-Fisher relation ($M_* \propto V_{\text{rot}}^{\alpha}$, $\alpha \sim 3$-$4$) connects disk galaxy luminosity to rotation velocity. The baryonic TFR is tighter and more fundamental.

2. The Faber-Jackson relation ($L \propto \sigma^{\gamma}$, $\gamma \sim 4$) connects elliptical galaxy luminosity to velocity dispersion, but has significant scatter.

3. The Fundamental Plane reduces the scatter by adding effective radius as a third variable. The "tilt" away from the virial prediction encodes $M/L$ variations.

4. These scaling relations provide constraints on galaxy formation models and are powerful distance indicators (used for peculiar velocity measurements and the Hubble constant).